In [2]:
import pandas as pd

from rdkit import Chem

from ground_truth.highlight_smarts import highlight_atoms_in_mol
from krfp_models import krfp_models, model_name_to_publication_name

In [3]:
disagreement_dict = {}
ds_lengths = {}

for model, pre_smarts, post_smarts in krfp_models:
    print(f"Model: {model.__class__.__name__}")
    print(f"Pre-SMARTS: {pre_smarts}")
    print(f"Post-SMARTS: {post_smarts}")

    publication_name = model_name_to_publication_name[model.__class__.__name__]
    disagreement_dict[publication_name] = []

    pre_mol = Chem.MolFromSmarts(pre_smarts)
    post_mol = Chem.MolFromSmarts(post_smarts)
    test_smiles = pd.read_csv(
        f"artifacts/explainability_method_tester/R2/{model.__class__.__name__}/positive_explainability_results.csv")[
        "SMILES"].tolist()

    ds_lengths[publication_name] = len(test_smiles)

    for smiles in test_smiles:
        mol = Chem.MolFromSmiles(smiles)
        pre_matches = mol.GetSubstructMatches(pre_mol)
        post_matches = mol.GetSubstructMatches(post_mol)

        assert not (pre_matches and not post_matches), f"Pre-matches found but no post-matches for molecule {smiles}."

        if not pre_matches and post_matches:
            disagreement_dict[publication_name].append((smiles, "BIG DIFFERENCE"))
            continue

        pre_highlighted = set()

        for match in pre_matches:
            pre_highlighted.update(match)

        post_highlighted = highlight_atoms_in_mol(mol, post_smarts)
        post_highlighted = set(post_highlighted)

        assert pre_highlighted.issubset(
            post_highlighted), f"Pre-highlighted atoms {pre_highlighted} are not subset of post-highlighted for molecule {smiles}."

        if pre_highlighted != post_highlighted:
            disagreement_dict[publication_name].append((smiles, "NOT-EQUAL-ATTRIBUTIONS"))

Model: MoleculeCDetector
Pre-SMARTS: [!#1][CH]([NH]C(=O)[!#1])C(Cl)(Cl)Cl
Post-SMARTS: [CHD3]([NH][CD3](=O))C(Cl)(Cl)Cl
Model: MoleculeEDetector
Pre-SMARTS: [!#1][CH]([!#1])C(=O)[CH2]C([CH3])([CH3])[CH2]C([!#1])([!#1])[!#1]
Post-SMARTS: [CHD3]C(=O)[CH2]C([CH3])([CH3])[CH2][CD4]
Model: MoleculeFDetector
Pre-SMARTS: [!#1][CH]([!#1])C(=O)[NH][NH]C(=O)[CH2][CH2][CH2][CH2][CH2][CH2][CH3]
Post-SMARTS: [CHD3]C(=O)[NH][NH]C(=O)[CH2][CH2][CH2][CH2][CH2][CH2][CH3]
Model: MoleculeGDetector
Pre-SMARTS: [!#1][CH]([!#1])C(=O)O[CH2][CH2]N([CH2][CH3])[CH2][CH3]
Post-SMARTS: [CHD3]C(=O)O[CH2][CH2]N([CH2][CH3])[CH2][CH3]
Model: MoleculeHDetector
Pre-SMARTS: [!#1][CH]([CH2]C(=N[NH]C(=O)[!#1])[!#1])C(=O)[OH]
Post-SMARTS: [CHD3]([CH2][CD3](=N[NH][CD3](=O)))C(=O)[OH]
Model: MoleculeIDetector
Pre-SMARTS: [!#1][CH]([!#1])[CH]([!#1])[!#1]
Post-SMARTS: [CHD3][CHD3]
Model: MoleculeJDetector
Pre-SMARTS: [!#1][CH]([!#1])[CH2][CH2][CH3]
Post-SMARTS: [CHD3][CH2][CH2][CH3]
Model: MoleculeKDetector
Pre-SMARTS: [!#1][C

In [6]:
for publication_name, disagreements in disagreement_dict.items():
    print(f"Publication: {publication_name}")
    print(f"Number of disagreements: {len(disagreements)} / {ds_lengths[publication_name]}")
    print(f"Disagreement %: {(len(disagreements) / ds_lengths[publication_name]) * 100:.2f}%")
    print("====")

Publication: Pattern 1
Number of disagreements: 1 / 7674
Disagreement %: 0.01%
====
Publication: Pattern 2
Number of disagreements: 184 / 248
Disagreement %: 74.19%
====
Publication: Pattern 3
Number of disagreements: 0 / 24
Disagreement %: 0.00%
====
Publication: Pattern 4
Number of disagreements: 2 / 1146
Disagreement %: 0.17%
====
Publication: Pattern 5
Number of disagreements: 10 / 34
Disagreement %: 29.41%
====
Publication: Pattern 6
Number of disagreements: 591 / 10000
Disagreement %: 5.91%
====
Publication: Pattern 7
Number of disagreements: 0 / 10000
Disagreement %: 0.00%
====
Publication: Pattern 8
Number of disagreements: 106 / 10000
Disagreement %: 1.06%
====
Publication: Pattern 9
Number of disagreements: 5 / 10000
Disagreement %: 0.05%
====
Publication: Pattern 10
Number of disagreements: 0 / 10000
Disagreement %: 0.00%
====
Publication: Pattern 11
Number of disagreements: 0 / 10000
Disagreement %: 0.00%
====
Publication: Pattern 12
Number of disagreements: 0 / 10000
Disag